In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296)

# data (as pandas dataframes)
X = diabetes_130_us_hospitals_for_years_1999_2008.data.features.copy()
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets.copy()

# metadata
print(diabetes_130_us_hospitals_for_years_1999_2008.metadata)

# variable information
print(diabetes_130_us_hospitals_for_years_1999_2008.variables)


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


{'uci_id': 296, 'name': 'Diabetes 130-US Hospitals for Years 1999-2008', 'repository_url': 'https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008', 'data_url': 'https://archive.ics.uci.edu/static/public/296/data.csv', 'abstract': 'The dataset represents ten years (1999-2008) of clinical care at 130 US hospitals and integrated delivery networks. Each row concerns hospital records of patients diagnosed with diabetes, who underwent laboratory, medications, and stayed up to 14 days. The goal is to determine the early readmission of the patient within 30 days of discharge.\nThe problem is important for the following reasons. Despite high-quality evidence showing improved clinical outcomes for diabetic patients who receive various preventive and therapeutic interventions, many patients do not receive them. This can be partially attributed to arbitrary diabetes management in hospital environments, which fail to attend to glycemic control. Failure to provide pro

In [3]:
import pandas as pd
import numpy as np

df = X.copy()
df['readmitted'] = y['readmitted'].values

df.head()

,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
df['readmitted'] = (df['readmitted'] == '<30').astype(int)

print('Readmitted distribution:')
print(df['readmitted'].value_counts())
print(f'Readmitted rate: {df["readmitted"].mean()}')

Readmitted distribution:
readmitted
0    90409
1    11357
Name: count, dtype: int64
Readmitted rate: 0.11159915885462728


In [5]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

# Label encode categorical columns
object_columns = df.select_dtypes(include='object')
object_column_names = object_columns.columns
object_column_list = object_column_names.tolist()
cat_cols = []
for col in object_column_list:
    if col != 'readmitted':
        cat_cols.append(col)

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

/tmp/ipykernel_8541/746806656.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipykernel_8541/746806656.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2,0,0,8,6,25,1,1,7,37,...,0,1,1,0,0,0,0,1,0,0
1,2,0,1,8,1,1,7,3,7,18,...,0,3,1,0,0,0,0,0,1,0
2,0,0,2,8,1,1,7,2,7,18,...,0,1,1,0,0,0,0,1,1,0
3,2,1,3,8,1,1,7,2,7,18,...,0,3,1,0,0,0,0,0,1,0
4,2,1,4,8,1,1,7,1,7,18,...,0,2,1,0,0,0,0,0,1,0


In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_model = df.drop(columns=['readmitted'])
y_model = df['readmitted']

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42, stratify=y_model
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]}')
print(f'Test: {X_test.shape[0]}')

Train: 81412
Test: 20354


In [7]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=100,
    random_state=42,
    early_stopping=True,
    verbose=True
)

mlp.fit(X_train_scaled, y_train)

Iteration 1, loss = 0.35306118
Validation score: 0.888234
Iteration 2, loss = 0.33785287
Validation score: 0.888479
Iteration 3, loss = 0.33527219
Validation score: 0.888479
Iteration 4, loss = 0.33351786
Validation score: 0.888479
Iteration 5, loss = 0.33222084
Validation score: 0.888357
Iteration 6, loss = 0.33099433
Validation score: 0.888234
Iteration 7, loss = 0.33010744
Validation score: 0.888602
Iteration 8, loss = 0.32881403
Validation score: 0.887865
Iteration 9, loss = 0.32772925
Validation score: 0.888479
Iteration 10, loss = 0.32700777
Validation score: 0.887988
Iteration 11, loss = 0.32620064
Validation score: 0.887988
Iteration 12, loss = 0.32519550
Validation score: 0.888111
Iteration 13, loss = 0.32418505
Validation score: 0.888234
Iteration 14, loss = 0.32336099
Validation score: 0.888111
Iteration 15, loss = 0.32243349
Validation score: 0.887865
Iteration 16, loss = 0.32165506
Validation score: 0.888111
Iteration 17, loss = 0.32063044
Validation score: 0.887620
Iterat

MLPClassifier(early_stopping=True, hidden_layer_sizes=(64, 32), max_iter=100,
              random_state=42, verbose=True)

In [8]:
y_pred = mlp.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted within 30 days']))

                           precision    recall  f1-score   support

           Not Readmitted       0.89      1.00      0.94     18083
Readmitted within 30 days       0.58      0.00      0.01      2271

                 accuracy                           0.89     20354
                macro avg       0.73      0.50      0.48     20354
             weighted avg       0.85      0.89      0.84     20354

